[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/01_hello_qwen_tokenizer.ipynb)

# ០១ · សួស្តី Tokenizer របស់ Qwen

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## តើ Tokenizer គឺជាអ្វី?

LLM មិនអាន «អក្សរ» ឬ «ពាក្យ» ទេ — វាអាន **token** (លេខសម្គាល់ផ្នែកនៃពាក្យ)។
Tokenizer បំបែកអត្ថបទ → token ids និងបកប្រែត្រឡប់វិញ។

សម្រាប់ភាសាខ្មែរ ចំណុចពិសេស៖
- ខ្មែរ **គ្មានដកឃ្លា**រវាងពាក្យ → tokenizer (BPE) បំបែកតាមក្រុមអក្សរ
- Qwen3 vocab = **151,936** tokens · Qwen3.5 vocab = **248,320** tokens (២០១ ភាសា)
- vocab ធំជាង → អក្សរខ្មែរបំបែកតូចជាង → អត្ថបទខ្មែរចំណាយ token តិចជាង

### សាកល្បងគិតដូច tokenizer (គ្មាន library ធំៗ!)

Cell ខាងក្រោមប្រើ Python សុទ្ធ — រត់បានគ្រប់ម៉ាស៊ីន។

In [ ]:
text = "សួស្តី ខ្ញុំឈ្មោះ សុខា"
print("ប្រវែងជាអក្សរ :", len(text))
print("បំបែកតាមដកឃ្លា:", text.split())
print("បំបែកតាមអក្សរ  :", " | ".join(list(text.replace(" ", ""))))

# ចំនួន token ពិតប្រហែលសម្រាប់ខ្មែរ: ~1.5–2.5 អក្សរ/token
est_tokens = len(text.replace(" ", "")) / 2
print(f"ការប៉ាន់ស្មាន token (ខ្មែរ): ~{est_tokens:.0f}")

## Tokenizer ពិតរបស់ Qwen (ត្រូវការ download ម៉ូដែល)

Cell ខាងក្រោម **guard** ដោយ environment variable — វាទាញយក tokenizer
(~១០ MB ប៉ុណ្ណោះ មិនមែន weights ទេ)។ ដើម្បីរត់ពិត៖

```bash
pip install "transformers>=5.8,<6"
export QWEN_FT_ALLOW_HEAVY=1
```

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # ផ្ទៀងផ្ទាត់ vocab + ចំនួន token សម្រាប់អក្សរខ្មែរ
    from transformers import AutoTokenizer

    for mid in ["Qwen/Qwen3-0.6B", "Qwen/Qwen3.5-0.8B"]:
        tok = AutoTokenizer.from_pretrained(mid)
        ids = tok.encode(text)
        print(f"{mid}: vocab={tok.vocab_size:,} · «{text}» → {len(ids)} tokens")
        print("   tokens:", tok.tokenize(text)[:12], "...")
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## អ្វីដែលយើងរៀន

1. Token ≠ អក្សរ ≠ ពាក្យ — វាជាឯកតាដែល tokenizer កំណត់
2. `max_length` ក្នុង config ហ្វឹក គិតជា **token** — ប្រើ tokenizer ពិនិត្យប្រវែងពិត
3. Qwen3.5 មាន vocab ធំជាង → ខ្មែរ efficient ជាង Qwen3

### លំហាត់
- Tokenize អត្ថបទខ្មែរ ៣ ប្រភេទ (ខ្លី/មធ្យម/វែង) ហើយគណនា token/អក្សរ ratio
- ប្រៀបធៀបចំនួន token រវាងអត្ថបទខ្មែរ និងអង់គ្លេសប្រវែង «ដូចគ្នា» (ជាពាក្យ)
- អានជំពូក [៣ · ម៉ូដែល និង Tokenizer](../chapters/ch03-models-and-tokenizer.md)